In [1]:
%pip install torch torchvision segmentation-models-pytorch albumentations scikit-learn pandas tqdm scikit-image torchstain

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp

# ==========================================
# ⚙️ SYSTEM-LEVEL GPU OPTIMIZATION
# ==========================================
torch.backends.cudnn.benchmark = True 
torch.set_float32_matmul_precision('high') # 🚀 Enables TF32 for RTX A6000

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "unet_finetuned_gbm.pth" 
DATA_DIR = "MNG_3.0"                 
MASK_DIR = "MNG_3.0_GroundTruth"      
OUT_DIR = "Results_GBM_Model_on_ALL_MNG"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "masks"), exist_ok=True)

# ==========================================
# 📏 DEFINING MISSING TRANSFORMS & METRICS
# ==========================================
# Fixed: Define val_tfms here so the loader can find it
val_tfms = A.Compose([
    A.Normalize(),
    ToTensorV2()
])

def get_patch_metrics(preds, targets, eps=1e-7):
    preds = (torch.sigmoid(preds) > 0.5).float()
    p, t = preds.view(preds.size(0), -1), targets.view(targets.size(0), -1)
    intersection = (p * t).sum(dim=1)
    total = p.sum(dim=1) + t.sum(dim=1)
    dice = (2. * intersection + eps) / (total + eps)
    iou = (intersection + eps) / (total - intersection + eps)
    return dice.detach().cpu().numpy(), iou.detach().cpu().numpy()

# ==========================================
# 🏗️ DATASET CLASS
# ==========================================
class AllDataDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        self.fnames = sorted([f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg', '.tif'))])
        self.transform = transform

    def __len__(self): return len(self.fnames)

    def __getitem__(self, idx):
        fn = self.fnames[idx]
        img = io.imread(os.path.join(self.img_dir, fn))[..., :3]
        mask = (io.imread(os.path.join(self.mask_dir, fn), as_gray=True) > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"].unsqueeze(0)
        return img, mask, fn

# ==========================================
# 🏗️ MODEL & DATA LOADER
# ==========================================
model = smp.Unet("resnet34", in_channels=3, classes=1).to(DEVICE)
# Handle models saved with torch.compile if necessary
sd = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in sd.items()})
model.eval()

# 🚀 Optimized Loader for A6000
loader = DataLoader(
    AllDataDataset(DATA_DIR, MASK_DIR, val_tfms), 
    batch_size=64, 
    num_workers=8, 
    pin_memory=True
)

# ==========================================
# 🚀 INFERENCE LOOP (WITH CUDA OPTIMIZATION)
# ==========================================
patch_metrics, batch_metrics = [], []



with torch.no_grad(), torch.amp.autocast("cuda"):
    for b_idx, (imgs, masks, fnames) in enumerate(tqdm(loader, desc="GBM->MNG Inference")):
        imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        
        preds = model(imgs)
        d_scores, i_scores = get_patch_metrics(preds, masks)
        
        # 1. Patch-wise results
        for d, i, fn in zip(d_scores, i_scores, fnames):
            patch_metrics.append({"filename": fn, "dice": d, "iou": i})
        
        # 2. Batch-wise results
        batch_metrics.append({
            "batch_id": b_idx, 
            "mean_dice": np.mean(d_scores), 
            "mean_iou": np.mean(i_scores)
        })
        
        # 3. Save Masks using GPU-accelerated processing
        binary = (torch.sigmoid(preds) > 0.5).cpu().numpy().astype(np.uint8) * 255
        for i in range(len(fnames)):
            io.imsave(os.path.join(OUT_DIR, "masks", fnames[i]), binary[i, 0], check_contrast=False)

# ==========================================
# 📊 EXPORT RESULTS
# ==========================================
pd.DataFrame(patch_metrics).to_csv(os.path.join(OUT_DIR, "all_patch_metrics.csv"), index=False)
pd.DataFrame(batch_metrics).to_csv(os.path.join(OUT_DIR, "all_batch_metrics.csv"), index=False)

print(f"\n✅ Finished processing {len(patch_metrics)} patches.")
print(f"📊 Global Mean Dice: {np.mean([p['dice'] for p in patch_metrics]):.4f}")
print(f"📊 Global Mean IoU:  {np.mean([p['iou'] for p in patch_metrics]):.4f}")

GBM->MNG Inference: 100%|██████████| 141/141 [00:20<00:00,  6.85it/s]


✅ Finished processing 8994 patches.
📊 Global Mean Dice: 0.2671
📊 Global Mean IoU:  0.1869


In [4]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp

# ==========================================
# ⚙️ SYSTEM-LEVEL GPU OPTIMIZATION
# ==========================================
torch.backends.cudnn.benchmark = True 
torch.set_float32_matmul_precision('high') # 🚀 Enables TF32 for RTX A6000

# ==========================================
# ⚙️ CONFIGURATION: MNG Model -> GBM Data
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "unet_finetuned_mng.pth"  # Your saved MNG model
DATA_DIR = "GBM_0067_0108"                # Path to your GBM images
MASK_DIR = "GBM_0067_0108_GroundTruth"           # Path to your GBM masks
OUT_DIR = "Results_MNG_Model_on_ALL_GBM"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "masks"), exist_ok=True)

# ==========================================
# 📏 TRANSFORMS & METRICS
# ==========================================
val_tfms = A.Compose([
    A.Normalize(),
    ToTensorV2()
])

def get_patch_metrics(preds, targets, eps=1e-7):
    preds = (torch.sigmoid(preds) > 0.5).float()
    p, t = preds.view(preds.size(0), -1), targets.view(targets.size(0), -1)
    intersection = (p * t).sum(dim=1)
    total = p.sum(dim=1) + t.sum(dim=1)
    dice = (2. * intersection + eps) / (total + eps)
    iou = (intersection + eps) / (total - intersection + eps)
    return dice.detach().cpu().numpy(), iou.detach().cpu().numpy()

# ==========================================
# 🏗️ DATASET CLASS
# ==========================================
class AllDataDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        # Filtering for common image extensions
        self.fnames = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.tif', '.jpeg'))])
        self.transform = transform

    def __len__(self): return len(self.fnames)

    def __getitem__(self, idx):
        fn = self.fnames[idx]
        img = io.imread(os.path.join(self.img_dir, fn))[..., :3]
        mask = (io.imread(os.path.join(self.mask_dir, fn), as_gray=True) > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"].unsqueeze(0)
        return img, mask, fn

# ==========================================
# 🏗️ MODEL & DATA LOADER
# ==========================================
# Re-initializing architecture
model = smp.Unet("resnet34", in_channels=3, classes=1).to(DEVICE)

# Load state dict and handle 'torch.compile' prefix if present
sd = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in sd.items()})
model.eval()

# Optimized Loader for A6000 (Batch Size 64)
loader = DataLoader(
    AllDataDataset(DATA_DIR, MASK_DIR, val_tfms), 
    batch_size=64, 
    num_workers=8, 
    pin_memory=True
)

# ==========================================
# 🚀 INFERENCE LOOP
# ==========================================
patch_metrics, batch_metrics = [], []



with torch.no_grad(), torch.amp.autocast("cuda"):
    for b_idx, (imgs, masks, fnames) in enumerate(tqdm(loader, desc="MNG->GBM Inference")):
        imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        
        preds = model(imgs)
        d_scores, i_scores = get_patch_metrics(preds, masks)
        
        # 1. Patch-wise logging
        for d, i, fn in zip(d_scores, i_scores, fnames):
            patch_metrics.append({"filename": fn, "dice": d, "iou": i})
        
        # 2. Batch-wise logging
        batch_metrics.append({
            "batch_id": b_idx, 
            "mean_dice": np.mean(d_scores), 
            "mean_iou": np.mean(i_scores)
        })
        
        # 3. Save predicted masks
        binary = (torch.sigmoid(preds) > 0.5).cpu().numpy().astype(np.uint8) * 255
        for i in range(len(fnames)):
            io.imsave(os.path.join(OUT_DIR, "masks", fnames[i]), binary[i, 0], check_contrast=False)

# ==========================================
# 📊 EXPORT RESULTS
# ==========================================
pd.DataFrame(patch_metrics).to_csv(os.path.join(OUT_DIR, "all_patch_metrics.csv"), index=False)
pd.DataFrame(batch_metrics).to_csv(os.path.join(OUT_DIR, "all_batch_metrics.csv"), index=False)

print(f"\n✅ Finished processing {len(patch_metrics)} GBM patches using MNG model.")
print(f"📊 Global Mean Dice: {np.mean([p['dice'] for p in patch_metrics]):.4f}")
print(f"📊 Global Mean IoU:  {np.mean([p['iou'] for p in patch_metrics]):.4f}")

MNG->GBM Inference: 100%|██████████| 1543/1543 [03:51<00:00,  6.66it/s]



✅ Finished processing 98705 GBM patches using MNG model.
📊 Global Mean Dice: 0.7371
📊 Global Mean IoU:  0.6470
